In [3]:
import pandas as pd
import numpy as np
import random

# Stop numbers from randomly changing every time we run the code
np.random.seed(42)
random.seed(42)

# Names and states for generating realistic clients
first_names = ['John', 'Jane', 'Michael', 'Emily', 'David', 'Sarah', 'James', 'Jessica', 'Robert', 'Lisa', 'William', 'Karen', 'Joseph', 'Nancy']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Rodriguez', 'Martinez', 'Lopez', 'Wilson']
states = ['NY', 'CA', 'TX', 'FL', 'IL', 'PA', 'OH', 'GA', 'NC', 'MI']
policy_types = ['Auto', 'Home', 'Life']

n_rows = 5000

# Base customer information setup
data = {
    'Policy_ID': [f'POL{100000 + i}' for i in range(n_rows)],
    'Customer_Name': [f"{random.choice(first_names)} {random.choice(last_names)}" for _ in range(n_rows)],
    'Age': np.random.randint(18, 76, size=n_rows),
    'Gender': [random.choice(['Male', 'Female']) for _ in range(n_rows)],
    'State': [random.choice(states) for _ in range(n_rows)],
    'Policy_Type': [random.choice(policy_types) for _ in range(n_rows)],
    'Annual_Premium': np.round(np.random.uniform(500, 3500, size=n_rows), 2),
    'Risk_Score': np.random.randint(10, 100, size=n_rows),  # Higher score = riskier client
    'Tenure_Months': np.random.randint(1, 120, size=n_rows)
}

# Industry issue 1: Customer churn (cancellations)
churn_prob = (data['Risk_Score'] / 200) + (1 - (data['Tenure_Months'] / 120)) * 0.4
data['Churned'] = [1 if random.random() < min(p, 0.95) else 0 for p in churn_prob]

# Industry issue 2: Claims fraud and large financial leaks
data['Claim_Filed'] = [1 if random.random() < 0.25 else 0 for _ in range(n_rows)]
claim_amounts = []
for i in range(n_rows):
    if data['Claim_Filed'][i] == 1:
        base_claim = data['Annual_Premium'][i] * np.random.uniform(1.5, 10)
        if data['Risk_Score'][i] > 80 and random.random() < 0.3:
            base_claim *= 2  # Simulating high-risk anomalous claims
        claim_amounts.append(round(base_claim, 2))
    else:
        claim_amounts.append(0.0)
data['Claim_Amount'] = claim_amounts

# Industry issue 3: Low digital adoption among older clients
data['Digital_User'] = [1 if random.random() > (data['Age'][i] / 100) else 0 for i in range(n_rows)]

# Save the dataset directly into your data folder using your exact path
df = pd.DataFrame(data)
df.to_csv(r'C:\Users\temit\OneDrive\Documents\insurance_analytics_project\data\insurance_data.csv', index=False)
print("Success! Created 'insurance_data.csv' inside your data folder.")


Success! Created 'insurance_data.csv' inside your data folder.


In [8]:
#Question 1: Revenue Concentration
import pandas as pd

# Load the dataset from your specific folder path
df = pd.read_csv(r'C:\Users\temit\OneDrive\Documents\insurance_analytics_project\data\insurance_data.csv')

print("1. TOTAL PREMIUMS COLLECTED BY POLICY TYPE:")
premiums = df.groupby('Policy_Type')['Annual_Premium'].sum()
print(premiums.map('${:,.2f}'.format))


1. TOTAL PREMIUMS COLLECTED BY POLICY TYPE:
Policy_Type
Auto    $3,453,631.05
Home    $3,158,135.14
Life    $3,379,355.55
Name: Annual_Premium, dtype: object


In [9]:
#Question 2: Customer Retention

print("2. CUSTOMER CHURN (CANCELLATION) RATE BY POLICY TYPE:")
churn_rate = df.groupby('Policy_Type')['Churned'].mean() * 100
print(churn_rate.map('{:.2f}%'.format))


2. CUSTOMER CHURN (CANCELLATION) RATE BY POLICY TYPE:
Policy_Type
Auto    47.63%
Home    48.34%
Life    48.78%
Name: Churned, dtype: object


In [10]:
#Question 3: Fraud & Financial Leakage Exposure

print("3. AVG CLAIM VALUE BY CUSTOMER RISK PROFILE (High Risk > 80):")
high_risk_claims = df[df['Claim_Filed'] == 1].groupby(df['Risk_Score'] > 80)['Claim_Amount'].mean()
high_risk_claims.index = ['Low/Medium Risk Profile', 'High Risk Profile (>80)']
print(high_risk_claims.map('${:,.2f}'.format))


3. AVG CLAIM VALUE BY CUSTOMER RISK PROFILE (High Risk > 80):
Low/Medium Risk Profile    $11,415.90
High Risk Profile (>80)    $16,213.55
Name: Claim_Amount, dtype: object


In [11]:
#Question 4: Digital App Adoption Drop-Off
print("4. DIGITAL APP ADOPTION RATE BY AGE BRACKET:")
age_bins = [17, 30, 50, 100]
age_labels = ['Young Adults (18-30)', 'Mid-Age (31-50)', 'Seniors (51+)']
df['Age_Group'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels)
digital_usage = df.groupby('Age_Group', observed=False)['Digital_User'].mean() * 100
print(digital_usage.map('{:.2f}%'.format))


4. DIGITAL APP ADOPTION RATE BY AGE BRACKET:
Age_Group
Young Adults (18-30)    75.87%
Mid-Age (31-50)         59.03%
Seniors (51+)           40.04%
Name: Digital_User, dtype: object


In [12]:
#Question 5: Geographic Profitability (Loss Ratio)

print("5. CLAIMS-TO-PREMIUM LOSS RATIO BY STATE:")
loss_ratio = df.groupby('State')['Claim_Amount'].sum() / df.groupby('State')['Annual_Premium'].sum()
print(loss_ratio.sort_values(ascending=False).map('{:.2f}x revenue paid out'.format))


5. CLAIMS-TO-PREMIUM LOSS RATIO BY STATE:
State
NY    1.87x revenue paid out
PA    1.66x revenue paid out
CA    1.62x revenue paid out
TX    1.59x revenue paid out
GA    1.59x revenue paid out
FL    1.52x revenue paid out
MI    1.47x revenue paid out
NC    1.43x revenue paid out
OH    1.38x revenue paid out
IL    1.33x revenue paid out
dtype: object
